# Batch Normalization: Building Strong Intuitions

This notebook will teach you **batch normalization** through hands-on experiments and visualizations. By the end, you'll have strong intuitions about:

- Why batch normalization was invented
- How it works (and why it works)
- When to use it (and when NOT to use it)
- Practical considerations and trade-offs

## Learning Objectives

1. ✅ Understand the internal covariate shift problem
2. ✅ Implement batch normalization from scratch
3. ✅ Visualize how batch norm stabilizes training
4. ✅ Learn the pros/cons through experiments
5. ✅ Know when to use alternatives (Layer Norm, Group Norm, etc.)

Let's dive in!

In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

# Import shared utilities
from aiml_notebooks import get_device, set_seed

# Enable autoreload
%load_ext autoreload
%autoreload 2

# Set random seed for reproducibility
set_seed(42)

# Device setup
device = get_device()

# Plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Setup complete!")

---

# Part 1: The Problem - Why Do We Need Batch Normalization?

## The Internal Covariate Shift Problem

Imagine training a deep neural network. As the network learns, the parameters of earlier layers change. This means the **distribution of inputs** to later layers keeps changing during training.

**Analogy**: Imagine learning to catch a ball, but the ball's weight and size change randomly every few throws. You'd constantly need to re-adjust your technique!

This is called **internal covariate shift** - the inputs to each layer come from a "covariate" (input distribution) that keeps "shifting" during training.

### Why is this a problem?

1. **Unstable gradients**: Changing input distributions → changing gradient magnitudes
2. **Slow convergence**: Network wastes time adapting to shifting distributions
3. **Requires small learning rates**: Otherwise, the network becomes unstable
4. **Limits depth**: Deeper networks amplify this problem

Let's visualize this problem!

In [ ]:
# Create a simple deep network WITHOUT batch normalization
class DeepNetWithoutBN(nn.Module):
    def __init__(self, input_size=784, hidden_size=100, num_layers=5, output_size=10):
        super().__init__()
        self.layers = nn.ModuleList()
        
        # First layer
        self.layers.append(nn.Linear(input_size, hidden_size))
        
        # Hidden layers
        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_size, hidden_size))
        
        # Output layer
        self.output = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        activations = []  # Store activations for visualization
        
        for layer in self.layers:
            x = layer(x)
            activations.append(x.detach().cpu())
            x = F.relu(x)
        
        x = self.output(x)
        return x, activations

# Create model and random input
model_no_bn = DeepNetWithoutBN(num_layers=6).to(device)
x_test = torch.randn(128, 784).to(device)  # Batch of random inputs

# Forward pass
with torch.no_grad():
    output, activations = model_no_bn(x_test)

print(f"Model has {len(model_no_bn.layers)} hidden layers")
print(f"\nActivation statistics across layers (before ReLU):")
print("="*60)
for i, act in enumerate(activations):
    mean = act.mean().item()
    std = act.std().item()
    print(f"Layer {i+1}: mean={mean:>7.3f}, std={std:>6.3f}")

### 🤔 Reflection

Look at the mean and standard deviation across layers. Do you notice:
- The means drift away from zero?
- The standard deviations vary wildly between layers?

This is just the **initialization**. During training, these distributions will keep shifting, making learning unstable!

In [ ]:
# Visualize activation distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, act in enumerate(activations):
    ax = axes[i]
    act_np = act.flatten().numpy()
    
    ax.hist(act_np, bins=50, alpha=0.7, edgecolor='black')
    ax.axvline(act_np.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {act_np.mean():.2f}')
    ax.set_title(f'Layer {i+1} Activations (before ReLU)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Activation Value')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.suptitle('Activation Distributions WITHOUT Batch Normalization', fontsize=14, fontweight='bold', y=1.02)
plt.show()

print("\n⚠️ Notice how the distributions look very different across layers!")
print("This is the problem batch normalization solves.")

---

# Part 2: The Solution - How Batch Normalization Works

## The Core Idea

Batch normalization normalizes the activations of each layer to have:
- **Mean = 0**
- **Standard deviation = 1**

This happens **for each mini-batch** during training.

## The Algorithm (Simplified)

For a layer with activations $x$:

1. **Compute batch statistics**:
   - $\mu_B = \frac{1}{m} \sum_{i=1}^{m} x_i$ (batch mean)
   - $\sigma_B^2 = \frac{1}{m} \sum_{i=1}^{m} (x_i - \mu_B)^2$ (batch variance)

2. **Normalize**:
   - $\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$ (where $\epsilon$ is a small constant for numerical stability)

3. **Scale and shift** (learnable parameters):
   - $y_i = \gamma \hat{x}_i + \beta$

### Why the scale and shift?

The learnable parameters $\gamma$ (scale) and $\beta$ (shift) allow the network to **undo the normalization if needed**. This preserves the representational power of the network.

For example:
- If $\gamma = \sqrt{\sigma_B^2 + \epsilon}$ and $\beta = \mu_B$, the normalization is completely undone!
- The network can learn the optimal amount of normalization for each layer.

Let's implement this from scratch!

In [ ]:
# Batch Normalization from scratch
class BatchNorm1d:
    """
    Batch Normalization for 1D inputs (used in fully connected layers).
    
    During training:
    - Normalize using batch statistics
    - Update running statistics with exponential moving average
    
    During inference:
    - Use running statistics (not batch statistics!)
    """
    
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        self.training = True
        
        # Learnable parameters (initialized to identity transform)
        self.gamma = torch.ones(num_features)  # Scale
        self.beta = torch.zeros(num_features)  # Shift
        
        # Running statistics (for inference)
        self.running_mean = torch.zeros(num_features)
        self.running_var = torch.ones(num_features)
    
    def __call__(self, x):
        # x shape: (batch_size, num_features)
        
        if self.training:
            # TRAINING MODE: Use batch statistics
            batch_mean = x.mean(dim=0)  # Mean across batch dimension
            batch_var = x.var(dim=0, unbiased=False)  # Variance across batch
            
            # Normalize
            x_normalized = (x - batch_mean) / torch.sqrt(batch_var + self.eps)
            
            # Update running statistics (exponential moving average)
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * batch_mean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * batch_var
        
        else:
            # INFERENCE MODE: Use running statistics
            x_normalized = (x - self.running_mean) / torch.sqrt(self.running_var + self.eps)
        
        # Scale and shift (learnable transformation)
        out = self.gamma * x_normalized + self.beta
        
        return out
    
    def eval(self):
        self.training = False
    
    def train(self):
        self.training = True

print("✓ Batch Normalization implementation complete!")
print("\nKey takeaways:")
print("1. Training mode uses batch statistics")
print("2. Inference mode uses running statistics")
print("3. Learnable gamma and beta preserve representational power")

### Let's test our implementation with a simple example

In [ ]:
# Create a batch of data with non-zero mean and non-unit variance
x = torch.randn(32, 10) * 5 + 3  # mean≈3, std≈5

print("Input statistics:")
print(f"  Mean: {x.mean():.3f}")
print(f"  Std:  {x.std():.3f}")
print()

# Apply batch normalization
bn = BatchNorm1d(num_features=10)
x_normalized = bn(x)

print("After batch normalization:")
print(f"  Mean: {x_normalized.mean():.3f}")
print(f"  Std:  {x_normalized.std():.3f}")
print()

print("✓ Successfully normalized to mean≈0, std≈1!")

### 🤔 Reflection

Try changing the input distribution (different mean/std). Does batch normalization always normalize to mean≈0 and std≈1?

What happens if you:
- Increase the batch size?
- Use a batch size of 1?

**Hint**: Batch normalization works best with larger batch sizes!

---

# Part 3: Training vs Inference - A Critical Distinction

## The Key Difference

**Training mode**:
- Normalize using **batch statistics** (mean/var computed from current batch)
- Update running statistics with exponential moving average

**Inference mode**:
- Normalize using **running statistics** (accumulated during training)
- Provides consistent behavior regardless of batch size

## Why is this necessary?

Imagine deploying your model to production:
- You might process **one sample at a time** (batch size = 1)
- Computing batch statistics on a single sample would be meaningless!
- Running statistics provide stable, representative statistics

Let's see what happens if we forget to switch modes!

In [ ]:
# Simulate training: accumulate running statistics
bn = BatchNorm1d(num_features=10)
bn.train()  # Training mode

print("Simulating training (updating running statistics)...\n")

for i in range(100):
    x_train = torch.randn(32, 10) * 2 + 1  # Random batches during "training"
    _ = bn(x_train)

print("Running statistics after training:")
print(f"  Mean: {bn.running_mean.mean():.3f}")
print(f"  Var:  {bn.running_var.mean():.3f}")
print()

# Test with a single sample
x_test_single = torch.randn(1, 10) * 2 + 1

print("Test sample (single):")
print(f"  Input mean: {x_test_single.mean():.3f}")
print()

# WRONG: Using training mode with a single sample
bn.train()
out_train_mode = bn(x_test_single)
print("⚠️ WRONG - Using training mode with single sample:")
print(f"  Output mean: {out_train_mode.mean():.3f}")
print(f"  Output std:  {out_train_mode.std():.3f}")
print("  (Batch stats on 1 sample are meaningless!)\n")

# CORRECT: Using inference mode
bn.eval()
out_eval_mode = bn(x_test_single)
print("✓ CORRECT - Using inference mode:")
print(f"  Output mean: {out_eval_mode.mean():.3f}")
print(f"  Output std:  {out_eval_mode.std():.3f}")
print("  (Using stable running statistics)")

### 🚨 Critical Lesson

**ALWAYS** remember to:
- Call `model.train()` during training
- Call `model.eval()` during inference/testing

Forgetting this is one of the most common bugs when using batch normalization!

---

# Part 4: Building a Network WITH Batch Normalization

Let's create a deep network with batch normalization and compare it to our earlier network without batch norm.

In [ ]:
class DeepNetWithBN(nn.Module):
    def __init__(self, input_size=784, hidden_size=100, num_layers=5, output_size=10):
        super().__init__()
        self.layers = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        
        # First layer
        self.layers.append(nn.Linear(input_size, hidden_size))
        self.batch_norms.append(nn.BatchNorm1d(hidden_size))
        
        # Hidden layers
        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_size, hidden_size))
            self.batch_norms.append(nn.BatchNorm1d(hidden_size))
        
        # Output layer
        self.output = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        activations = []  # Store activations for visualization
        
        for layer, bn in zip(self.layers, self.batch_norms):
            x = layer(x)
            x = bn(x)  # Apply batch normalization
            activations.append(x.detach().cpu())
            x = F.relu(x)
        
        x = self.output(x)
        return x, activations

# Create model
model_with_bn = DeepNetWithBN(num_layers=6).to(device)
x_test = torch.randn(128, 784).to(device)

# Forward pass
with torch.no_grad():
    output, activations_bn = model_with_bn(x_test)

print(f"Model has {len(model_with_bn.layers)} hidden layers (with batch norm)")
print(f"\nActivation statistics across layers (after batch norm, before ReLU):")
print("="*60)
for i, act in enumerate(activations_bn):
    mean = act.mean().item()
    std = act.std().item()
    print(f"Layer {i+1}: mean={mean:>7.3f}, std={std:>6.3f}")

print("\n✓ Notice how consistent the statistics are across layers!")

### Side-by-Side Comparison: With vs Without Batch Norm

In [ ]:
# Compare activation distributions
fig, axes = plt.subplots(2, 6, figsize=(18, 6))

# Without batch norm (top row)
for i, act in enumerate(activations):
    ax = axes[0, i]
    act_np = act.flatten().numpy()
    ax.hist(act_np, bins=50, alpha=0.7, edgecolor='black', color='red')
    ax.set_title(f'Layer {i+1}', fontsize=10)
    ax.set_xlim(-10, 10)
    if i == 0:
        ax.set_ylabel('Without BatchNorm', fontsize=11, fontweight='bold')

# With batch norm (bottom row)
for i, act in enumerate(activations_bn):
    ax = axes[1, i]
    act_np = act.flatten().numpy()
    ax.hist(act_np, bins=50, alpha=0.7, edgecolor='black', color='green')
    ax.set_xlim(-10, 10)
    if i == 0:
        ax.set_ylabel('With BatchNorm', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.suptitle('Activation Distributions: Without vs With Batch Normalization', 
             fontsize=14, fontweight='bold', y=1.02)
plt.show()

print("\n🎯 Key Observation:")
print("  - WITHOUT BatchNorm (red): Distributions vary wildly across layers")
print("  - WITH BatchNorm (green): Distributions are consistent (centered at 0)")

### 🤔 Reflection

Look at the difference between the two rows:
- **Top row (red)**: Distributions are all over the place
- **Bottom row (green)**: Distributions are consistent and centered

This stability is why batch normalization enables faster, more stable training!

---

# Part 5: Experiment 1 - Learning Rate Sensitivity

One of the biggest benefits of batch normalization is that it allows us to use **much higher learning rates**.

Let's train two networks (with and without batch norm) at different learning rates and see what happens!

In [ ]:
# Create synthetic dataset (simple binary classification)
def create_synthetic_data(n_samples=1000, n_features=20):
    X = torch.randn(n_samples, n_features)
    # Create non-linear decision boundary
    y = ((X[:, 0]**2 + X[:, 1]**2) > 1).long()
    return TensorDataset(X, y)

train_dataset = create_synthetic_data(n_samples=2000)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print(f"✓ Created synthetic dataset with {len(train_dataset)} samples")
print(f"  Features: {train_dataset[0][0].shape[0]}")
print(f"  Classes: 2 (binary classification)")

Create a simplified training function that we can reuse across different experiments.

In [ ]:
# Simplified training function
def train_model(model, train_loader, learning_rate, num_epochs=20):
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    
    losses = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass (handle different return types)
            output = model(X_batch)
            if isinstance(output, tuple):
                output = output[0]  # Get predictions from tuple
            
            loss = criterion(output, y_batch)
            
            # Check for NaN/Inf
            if torch.isnan(loss) or torch.isinf(loss):
                return losses  # Stop if training diverged
            
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        losses.append(epoch_loss / len(train_loader))
    
    return losses

print("✓ Training function ready")

Test how models with and without batch normalization handle different learning rates.

In [ ]:
# Test different learning rates
learning_rates = [0.001, 0.01, 0.1, 1.0]

results_without_bn = {}
results_with_bn = {}

print("Training models with different learning rates...\n")
print("="*60)

for lr in learning_rates:
    print(f"\nLearning Rate: {lr}")
    
    # Without batch norm
    set_seed(42)
    model_no_bn = DeepNetWithoutBN(input_size=20, hidden_size=50, num_layers=4, output_size=2).to(device)
    losses_no_bn = train_model(model_no_bn, train_loader, lr, num_epochs=20)
    results_without_bn[lr] = losses_no_bn
    
    if len(losses_no_bn) < 20:
        print(f"  Without BN: DIVERGED at epoch {len(losses_no_bn)}")
    else:
        print(f"  Without BN: Final loss = {losses_no_bn[-1]:.4f}")
    
    # With batch norm
    set_seed(42)
    model_with_bn = DeepNetWithBN(input_size=20, hidden_size=50, num_layers=4, output_size=2).to(device)
    losses_with_bn = train_model(model_with_bn, train_loader, lr, num_epochs=20)
    results_with_bn[lr] = losses_with_bn
    
    if len(losses_with_bn) < 20:
        print(f"  With BN:    DIVERGED at epoch {len(losses_with_bn)}")
    else:
        print(f"  With BN:    Final loss = {losses_with_bn[-1]:.4f}")

print("\n" + "="*60)
print("✓ Training complete!")

Visualize the learning curves to see which model is more stable across different learning rates.

In [ ]:
# Visualize learning curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, lr in enumerate(learning_rates):
    ax = axes[i]
    
    # Plot without batch norm
    if results_without_bn[lr]:
        ax.plot(results_without_bn[lr], 'r-', linewidth=2, label='Without BatchNorm', alpha=0.8)
    
    # Plot with batch norm
    if results_with_bn[lr]:
        ax.plot(results_with_bn[lr], 'g-', linewidth=2, label='With BatchNorm', alpha=0.8)
    
    ax.set_title(f'Learning Rate = {lr}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 2)  # Consistent scale

plt.tight_layout()
plt.suptitle('Learning Rate Sensitivity: With vs Without Batch Normalization', 
             fontsize=14, fontweight='bold', y=1.02)
plt.show()

print("\n🎯 Key Insights:")
print("  1. WITHOUT BatchNorm: High learning rates cause divergence")
print("  2. WITH BatchNorm: Can handle much higher learning rates")
print("  3. BatchNorm enables faster convergence with larger learning rates")

### 🤔 Reflection

Look at the learning curves:

1. **Low learning rates (0.001, 0.01)**: Both models work, but batch norm converges faster
2. **High learning rates (0.1, 1.0)**: Without batch norm, training often diverges
3. **With batch norm**: Training is stable even at very high learning rates

**Why?** Batch normalization reduces the sensitivity to weight initialization and learning rates by normalizing activations!

---

# Part 6: Experiment 2 - Network Depth

Batch normalization is especially important for **very deep networks**. Let's see what happens when we increase network depth!

In [ ]:
# Test different network depths
depths = [2, 5, 10, 20]
depth_results_without_bn = {}
depth_results_with_bn = {}

print("Training networks with different depths...\n")
print("="*60)

for depth in depths:
    print(f"\nDepth: {depth} layers")
    
    # Without batch norm
    set_seed(42)
    model_no_bn = DeepNetWithoutBN(input_size=20, hidden_size=50, num_layers=depth, output_size=2).to(device)
    losses_no_bn = train_model(model_no_bn, train_loader, learning_rate=0.01, num_epochs=30)
    depth_results_without_bn[depth] = losses_no_bn
    
    if len(losses_no_bn) < 30:
        print(f"  Without BN: DIVERGED at epoch {len(losses_no_bn)}")
    else:
        print(f"  Without BN: Final loss = {losses_no_bn[-1]:.4f}")
    
    # With batch norm
    set_seed(42)
    model_with_bn = DeepNetWithBN(input_size=20, hidden_size=50, num_layers=depth, output_size=2).to(device)
    losses_with_bn = train_model(model_with_bn, train_loader, learning_rate=0.01, num_epochs=30)
    depth_results_with_bn[depth] = losses_with_bn
    
    if len(losses_with_bn) < 30:
        print(f"  With BN:    DIVERGED at epoch {len(losses_with_bn)}")
    else:
        print(f"  With BN:    Final loss = {losses_with_bn[-1]:.4f}")

print("\n" + "="*60)
print("✓ Training complete!")

Compare how network depth affects training with and without batch normalization.

In [ ]:
# Visualize depth comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, depth in enumerate(depths):
    ax = axes[i]
    
    # Plot without batch norm
    if depth_results_without_bn[depth]:
        ax.plot(depth_results_without_bn[depth], 'r-', linewidth=2, label='Without BatchNorm', alpha=0.8)
    
    # Plot with batch norm
    if depth_results_with_bn[depth]:
        ax.plot(depth_results_with_bn[depth], 'g-', linewidth=2, label='With BatchNorm', alpha=0.8)
    
    ax.set_title(f'{depth} Hidden Layers', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 2)

plt.tight_layout()
plt.suptitle('Network Depth: With vs Without Batch Normalization', 
             fontsize=14, fontweight='bold', y=1.02)
plt.show()

print("\n🎯 Key Insights:")
print("  1. Shallow networks (2-5 layers): Both approaches work")
print("  2. Deep networks (10+ layers): Without BN struggles to train")
print("  3. BatchNorm enables training of very deep networks")

### 🤔 Reflection

As networks get deeper:
- **Without batch norm**: Training becomes increasingly unstable
- **With batch norm**: Training remains stable even at 20+ layers

This is why batch normalization was crucial for training very deep networks like ResNet (152 layers!).

---

# Part 7: Where to Place Batch Normalization?

There's been debate about where to place batch normalization:

**Option 1: Before activation** (original paper)
```
Linear → BatchNorm → ReLU
```

**Option 2: After activation**
```
Linear → ReLU → BatchNorm
```

The original paper placed it **before** the activation function, but both can work. Let's compare!

In [ ]:
class DeepNetBNAfterActivation(nn.Module):
    """Batch norm AFTER activation function"""
    def __init__(self, input_size=784, hidden_size=100, num_layers=5, output_size=10):
        super().__init__()
        self.layers = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        
        self.layers.append(nn.Linear(input_size, hidden_size))
        self.batch_norms.append(nn.BatchNorm1d(hidden_size))
        
        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_size, hidden_size))
            self.batch_norms.append(nn.BatchNorm1d(hidden_size))
        
        self.output = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        for layer, bn in zip(self.layers, self.batch_norms):
            x = layer(x)
            x = F.relu(x)  # Activation BEFORE batch norm
            x = bn(x)
        
        x = self.output(x)
        return x

# Compare both placements
print("Training with different batch norm placements...\n")

# Before activation (standard)
set_seed(42)
model_bn_before = DeepNetWithBN(input_size=20, hidden_size=50, num_layers=5, output_size=2).to(device)
losses_bn_before = train_model(model_bn_before, train_loader, learning_rate=0.01, num_epochs=30)
print(f"BatchNorm BEFORE activation: Final loss = {losses_bn_before[-1]:.4f}")

# After activation
set_seed(42)
model_bn_after = DeepNetBNAfterActivation(input_size=20, hidden_size=50, num_layers=5, output_size=2).to(device)
losses_bn_after = train_model(model_bn_after, train_loader, learning_rate=0.01, num_epochs=30)
print(f"BatchNorm AFTER activation:  Final loss = {losses_bn_after[-1]:.4f}")

Visualize the comparison between batch normalization placed before vs after the activation function.

In [ ]:
# Visualize comparison
plt.figure(figsize=(10, 6))
plt.plot(losses_bn_before, 'b-', linewidth=2, label='BatchNorm BEFORE activation', alpha=0.8)
plt.plot(losses_bn_after, 'orange', linewidth=2, label='BatchNorm AFTER activation', alpha=0.8)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('BatchNorm Placement Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.show()

print("\n🎯 Key Insight:")
print("  Both placements work! The original paper used 'before activation',")
print("  but recent research suggests 'after activation' can also be effective.")
print("  The best choice can depend on your specific architecture.")

---

# Part 8: Pros and Cons of Batch Normalization

Let's summarize what we've learned!

## ✅ Pros (Advantages)

1. **Faster training** - Can use higher learning rates
2. **More stable gradients** - Reduces internal covariate shift
3. **Less sensitive to initialization** - Xavier/He initialization less critical
4. **Regularization effect** - Slight noise from batch statistics acts as regularization
5. **Enables deeper networks** - Can train 100+ layer networks
6. **Reduces need for dropout** - The regularization effect can replace dropout in some cases

## ❌ Cons (Disadvantages)

1. **Batch size dependency** - Doesn't work well with very small batches
2. **Different behavior in training vs inference** - Must remember to switch modes
3. **Additional computation** - Adds overhead (usually small)
4. **Extra memory** - Stores running statistics
5. **Not suitable for online learning** - Requires mini-batches
6. **Can hurt performance in some cases** - Especially in recurrent networks

Let's explore some of these cons with experiments!

## Experiment: Batch Size Sensitivity

In [ ]:
# Test with different batch sizes
batch_sizes = [4, 16, 64, 256]
batch_size_results = {}

print("Testing batch normalization with different batch sizes...\n")
print("="*60)

for batch_size in batch_sizes:
    print(f"\nBatch Size: {batch_size}")
    
    # Create data loader with specific batch size
    loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    # Train with batch norm
    set_seed(42)
    model = DeepNetWithBN(input_size=20, hidden_size=50, num_layers=5, output_size=2).to(device)
    losses = train_model(model, loader, learning_rate=0.01, num_epochs=30)
    batch_size_results[batch_size] = losses
    
    print(f"  Final loss: {losses[-1]:.4f}")

print("\n" + "="*60)
print("✓ Training complete!")

See how different batch sizes affect the stability and performance of batch normalization.

In [ ]:
# Visualize batch size impact
plt.figure(figsize=(12, 6))

colors = ['red', 'orange', 'green', 'blue']
for (batch_size, losses), color in zip(batch_size_results.items(), colors):
    plt.plot(losses, linewidth=2, label=f'Batch Size = {batch_size}', alpha=0.8, color=color)

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Batch Size Impact on Batch Normalization', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.show()

print("\n🎯 Key Insight:")
print("  - Smaller batches (4, 16): Noisier training, batch statistics less representative")
print("  - Larger batches (64, 256): Smoother training, better batch statistics")
print("  - Recommendation: Use batch size >= 32 when possible")

### 🤔 Reflection

Notice how small batch sizes (4, 16) lead to noisier training? This is because:
- Small batches → less representative statistics
- Batch statistics have higher variance
- Running mean/variance accumulate noisy estimates

**When batch size = 1**: Batch normalization completely breaks down! (You can't compute meaningful statistics from a single sample)

---

# Part 9: When NOT to Use Batch Normalization

Batch normalization isn't always the right choice. Here are cases where you should avoid it or use alternatives:

## 1. Online Learning (Batch Size = 1)

If you need to process one sample at a time during training, batch normalization won't work.

**Alternative**: Layer Normalization

## 2. Recurrent Neural Networks (RNNs)

Batch normalization struggles with RNNs because:
- Different sequence lengths in the batch
- Temporal dependencies make batch statistics misleading

**Alternative**: Layer Normalization (used in LSTMs, Transformers)

## 3. Small Batch Sizes (< 16)

When hardware constraints force small batches, batch normalization becomes unstable.

**Alternative**: Group Normalization

## 4. Style Transfer / Generative Models

Batch statistics can interfere with capturing individual sample characteristics.

**Alternative**: Instance Normalization

Let's explore the alternatives!

---

# Part 10: Alternatives to Batch Normalization

The normalization family has grown beyond batch normalization!

In [ ]:
# Visualize different normalization approaches
from IPython.display import Markdown, display

display(Markdown("""
## Normalization Family Overview

| Type | Normalizes Over | Best For | PyTorch Class |
|------|----------------|----------|---------------|
| **Batch Norm** | Batch dimension | CNNs, MLPs with large batches | `nn.BatchNorm1d`, `nn.BatchNorm2d` |
| **Layer Norm** | Feature dimension | RNNs, Transformers, small batches | `nn.LayerNorm` |
| **Instance Norm** | Spatial dimensions per sample | Style transfer, GANs | `nn.InstanceNorm2d` |
| **Group Norm** | Groups of channels | Small batch sizes (1-32) | `nn.GroupNorm` |
"""))

print("\n" + "="*60)
print("Quick Reference Guide")
print("="*60)

Implement a network using layer normalization as an alternative to batch normalization.

In [ ]:
# Demonstrate Layer Normalization
class DeepNetWithLayerNorm(nn.Module):
    """Network with Layer Normalization (good for RNNs/Transformers)"""
    def __init__(self, input_size=20, hidden_size=50, num_layers=5, output_size=2):
        super().__init__()
        self.layers = nn.ModuleList()
        self.layer_norms = nn.ModuleList()
        
        self.layers.append(nn.Linear(input_size, hidden_size))
        self.layer_norms.append(nn.LayerNorm(hidden_size))
        
        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_size, hidden_size))
            self.layer_norms.append(nn.LayerNorm(hidden_size))
        
        self.output = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        for layer, ln in zip(self.layers, self.layer_norms):
            x = layer(x)
            x = ln(x)  # Layer normalization
            x = F.relu(x)
        x = self.output(x)
        return x

print("Layer Normalization vs Batch Normalization:")
print("="*60)
print("\nBatch Norm: Normalizes across the BATCH dimension")
print("  - Each feature is normalized using statistics from the entire batch")
print("  - Requires large batches for stable statistics")
print()
print("Layer Norm: Normalizes across the FEATURE dimension")
print("  - Each sample is normalized independently using its own feature statistics")
print("  - Works with batch size = 1!")
print("  - Perfect for RNNs and Transformers")

Test batch normalization vs layer normalization with very small batch sizes.

In [ ]:
# Compare with very small batch size
small_batch_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

print("\nComparing normalizations with SMALL batch size (4)...\n")

# Batch Norm
set_seed(42)
model_bn = DeepNetWithBN(input_size=20, hidden_size=50, num_layers=5, output_size=2).to(device)
losses_bn_small = train_model(model_bn, small_batch_loader, learning_rate=0.01, num_epochs=30)
print(f"Batch Norm:  Final loss = {losses_bn_small[-1]:.4f}")

# Layer Norm
set_seed(42)
model_ln = DeepNetWithLayerNorm(input_size=20, hidden_size=50, num_layers=5, output_size=2).to(device)
losses_ln_small = train_model(model_ln, small_batch_loader, learning_rate=0.01, num_epochs=30)
print(f"Layer Norm:  Final loss = {losses_ln_small[-1]:.4f}")

Compare the training curves for batch normalization vs layer normalization with small batches.

In [ ]:
# Visualize
plt.figure(figsize=(12, 6))
plt.plot(losses_bn_small, 'b-', linewidth=2, label='Batch Norm (struggles with small batch)', alpha=0.8)
plt.plot(losses_ln_small, 'green', linewidth=2, label='Layer Norm (designed for small batch)', alpha=0.8)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Batch Norm vs Layer Norm (Batch Size = 4)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.show()

print("\n🎯 Key Insight:")
print("  Layer Norm is more stable than Batch Norm when batch sizes are very small!")

---

# Part 11: Practical Decision Guide

Use this flowchart to decide which normalization to use:

```
START: Do you need normalization?
  |
  ├─> Are you training a CNN or MLP?
  │     |
  │     ├─> Batch size >= 32?
  │     │     ├─> YES → Use Batch Norm ✓
  │     │     └─> NO  → Use Group Norm ✓
  │     │
  ├─> Are you training an RNN or Transformer?
  │     └─> Use Layer Norm ✓
  │
  ├─> Are you doing style transfer?
  │     └─> Use Instance Norm ✓
  │
  └─> Batch size = 1 (online learning)?
        └─> Use Layer Norm or Group Norm ✓
```

---

# Part 12: Final Experiment - Real Classification Task (MNIST)

Let's put everything together with a real classification task!

In [ ]:
# Load MNIST dataset
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

# Download and load training data
mnist_train = datasets.MNIST(root='./tmp/data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root='./tmp/data', train=False, download=True, transform=transform)

# Create data loaders
train_loader_mnist = DataLoader(mnist_train, batch_size=128, shuffle=True)
test_loader_mnist = DataLoader(mnist_test, batch_size=128, shuffle=False)

print(f"✓ MNIST dataset loaded")
print(f"  Training samples: {len(mnist_train)}")
print(f"  Test samples: {len(mnist_test)}")
print(f"  Image shape: {mnist_train[0][0].shape}")
print(f"  Number of classes: 10")

Create an enhanced training function that includes validation accuracy tracking.

In [ ]:
# Enhanced training function with validation
def train_and_evaluate(model, train_loader, test_loader, learning_rate=0.01, num_epochs=10):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    
    train_losses = []
    test_accuracies = []
    
    for epoch in tqdm(range(num_epochs), desc="Training"):
        # Training
        model.train()
        epoch_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.view(X_batch.size(0), -1).to(device)  # Flatten images
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()
            
            output = model(X_batch)
            if isinstance(output, tuple):
                output = output[0]
            
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        train_losses.append(epoch_loss / len(train_loader))
        
        # Evaluation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.view(X_batch.size(0), -1).to(device)
                y_batch = y_batch.to(device)
                
                output = model(X_batch)
                if isinstance(output, tuple):
                    output = output[0]
                
                _, predicted = torch.max(output, 1)
                total += y_batch.size(0)
                correct += (predicted == y_batch).sum().item()
        
        accuracy = 100 * correct / total
        test_accuracies.append(accuracy)
    
    return train_losses, test_accuracies

print("✓ Training function ready")

Train both models on the MNIST dataset to see batch normalization in action on a real problem.

In [ ]:
# Train both models
print("Training on MNIST dataset...\n")
print("="*60)

# Without batch norm
print("\n1. Training WITHOUT Batch Normalization...")
set_seed(42)
model_mnist_no_bn = DeepNetWithoutBN(input_size=784, hidden_size=128, num_layers=4, output_size=10).to(device)
losses_no_bn, acc_no_bn = train_and_evaluate(model_mnist_no_bn, train_loader_mnist, test_loader_mnist, 
                                               learning_rate=0.001, num_epochs=10)
print(f"   Final accuracy: {acc_no_bn[-1]:.2f}%")

# With batch norm
print("\n2. Training WITH Batch Normalization...")
set_seed(42)
model_mnist_with_bn = DeepNetWithBN(input_size=784, hidden_size=128, num_layers=4, output_size=10).to(device)
losses_with_bn, acc_with_bn = train_and_evaluate(model_mnist_with_bn, train_loader_mnist, test_loader_mnist, 
                                                   learning_rate=0.001, num_epochs=10)
print(f"   Final accuracy: {acc_with_bn[-1]:.2f}%")

print("\n" + "="*60)
print("✓ MNIST training complete!")

Visualize the final training results comparing models with and without batch normalization.

In [ ]:
# Visualize final results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Training loss
ax1.plot(losses_no_bn, 'r-', linewidth=2, label='Without BatchNorm', marker='o', markersize=6)
ax1.plot(losses_with_bn, 'g-', linewidth=2, label='With BatchNorm', marker='o', markersize=6)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Training Loss', fontsize=12)
ax1.set_title('Training Loss Over Time', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)

# Test accuracy
ax2.plot(acc_no_bn, 'r-', linewidth=2, label='Without BatchNorm', marker='o', markersize=6)
ax2.plot(acc_with_bn, 'g-', linewidth=2, label='With BatchNorm', marker='o', markersize=6)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Test Accuracy (%)', fontsize=12)
ax2.set_title('Test Accuracy Over Time', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("Final Results Summary")
print("="*60)
print(f"\nWithout Batch Norm:")
print(f"  Final accuracy: {acc_no_bn[-1]:.2f}%")
print(f"  Epochs to reach 95%: {next((i+1 for i, acc in enumerate(acc_no_bn) if acc >= 95), 'Not reached')}")

print(f"\nWith Batch Norm:")
print(f"  Final accuracy: {acc_with_bn[-1]:.2f}%")
print(f"  Epochs to reach 95%: {next((i+1 for i, acc in enumerate(acc_with_bn) if acc >= 95), 'Not reached')}")

improvement = acc_with_bn[-1] - acc_no_bn[-1]
print(f"\nImprovement with BatchNorm: +{improvement:.2f}%")

---

# Part 13: Summary & Key Takeaways

## 🎓 What You've Learned

### 1. The Problem
- **Internal covariate shift**: Input distributions to layers keep changing during training
- Leads to unstable training, slow convergence, and limits network depth

### 2. The Solution
- **Batch Normalization**: Normalize layer inputs to zero mean and unit variance
- Uses learnable scale (γ) and shift (β) parameters to preserve representational power
- Different behavior in training vs inference mode

### 3. Benefits
- ✅ Faster training (can use higher learning rates)
- ✅ More stable gradients
- ✅ Less sensitive to initialization
- ✅ Slight regularization effect
- ✅ Enables training of very deep networks

### 4. Limitations
- ❌ Requires reasonable batch sizes (>= 16-32)
- ❌ Doesn't work well for online learning (batch size = 1)
- ❌ Not ideal for RNNs
- ❌ Must remember to switch between train/eval modes

### 5. When to Use Alternatives
- **Layer Norm**: RNNs, Transformers, small batches
- **Group Norm**: Very small batches (1-32)
- **Instance Norm**: Style transfer, generative models

## 🧠 Mental Model

Think of batch normalization as **"resetting the playing field"** for each layer:
- Instead of each layer adapting to constantly shifting inputs
- Each layer sees consistent, normalized inputs
- Allows the network to focus on learning the right transformations, not compensating for shifting distributions

## 📋 Practical Checklist

When using batch normalization:

- [ ] Use batch size >= 32 when possible
- [ ] Place batch norm before or after activation (both can work)
- [ ] Always call `model.train()` during training
- [ ] Always call `model.eval()` during inference
- [ ] Can often use higher learning rates (experiment!)
- [ ] Consider alternatives for RNNs or very small batches

## 🚀 Next Steps

Try these experiments:

1. **Different architectures**: Try batch norm with CNNs (use `nn.BatchNorm2d`)
2. **Residual connections**: Combine batch norm with skip connections (ResNet-style)
3. **Group normalization**: Implement and compare with batch norm on small batches
4. **Visualization**: Plot the learned γ and β parameters to see what the network learns

## 📚 Further Reading

- Original paper: [Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift](https://arxiv.org/abs/1502.03167)
- Layer Normalization: [Layer Normalization](https://arxiv.org/abs/1607.06450)
- Group Normalization: [Group Normalization](https://arxiv.org/abs/1803.08494)

---

# 🎉 Congratulations!

You now have strong intuitions about batch normalization:

- ✅ You understand **why** it was invented
- ✅ You know **how** it works (and implemented it from scratch!)
- ✅ You've seen **when** to use it (and when NOT to)
- ✅ You understand the **trade-offs** through experiments

Go forth and normalize! 🚀